# Train the skin lesion classifier on a GPU

This notebook runs the training pipeline in `ml/` end to end: download HAM10000, split by lesion,
fine-tune, calibrate, evaluate once on the held-out test set, and export an ONNX model for the API.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (or better). A full run takes about an hour on a T4.
2. Kaggle credentials: add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab secrets (key icon in the left
   sidebar), or upload `kaggle.json` when asked.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/alimehdikhan/Cancer-Detection-Model.git"
BRANCH = "main"

!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/project
%cd /content/project/ml
!pip install -q -r requirements.txt

## 2. Kaggle credentials

In [ ]:
import os

try:
    from google.colab import userdata

    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Using Kaggle credentials from Colab secrets.")
except Exception:
    from google.colab import files

    print("Upload kaggle.json (Kaggle → Settings → API → Create New Token)")
    uploaded = files.upload()
    path = os.path.expanduser("~/.kaggle/kaggle.json")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        f.write(uploaded["kaggle.json"])
    os.chmod(path, 0o600)

## 3. Download HAM10000 (~5 GB)

In [ ]:
!python -m skin_ml.download --data-dir /content/data/ham10000

## 4. Train, calibrate, evaluate and export

Defaults: EfficientNet-B2 at 256 px (fast to serve on CPU), up to 30 epochs with early stopping on
validation balanced accuracy. `python -m skin_ml.train --help` lists every option. For an
accuracy-first model that is slower to serve, try `--backbone efficientnet_b3.ra2_in1k --image-size 288`.

In [ ]:
!python -m skin_ml.train --data-dir /content/data/ham10000 --output-dir /content/runs/effnet_b2 --num-workers 2

## 5. Results on the held-out test set

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

m = json.load(open("/content/runs/effnet_b2/metrics.json"))
for key in ["test_images", "accuracy", "balanced_accuracy", "macro_f1", "auc_macro", "melanoma_recall"]:
    print(f"{key:>18}: {m[key]:.4f}" if isinstance(m[key], float) else f"{key:>18}: {m[key]}")

cm = np.array(m["confusion_matrix"], dtype=float)
norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(m["classes"])), m["classes"])
ax.set_yticks(range(len(m["classes"])), m["classes"])
ax.set_xlabel("predicted")
ax.set_ylabel("true")
for i in range(len(norm)):
    for j in range(len(norm)):
        ax.text(j, i, f"{norm[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if norm[i, j] > 0.5 else "black")
ax.set_title("Test set confusion matrix (each row sums to 1)")
fig.colorbar(im)
plt.show()

## 6. Download the model

Unzip `model.zip` into `models/current/` in the repository and restart the API
(or `docker compose restart api`). The web app's model card will show these test-set metrics.

In [ ]:
!cd /content/runs/effnet_b2/export && zip -q /content/model.zip model.onnx model.json metrics.json
from google.colab import files

files.download("/content/model.zip")